In [1]:
import xarray as xr
import os   
import parflow as pf
import plotly.express as px
from parflow.tools.hydrology import calculate_overland_flow_grid, calculate_subsurface_storage, calculate_water_table_depth
import numpy as np
import shutil
import json
import plotly.io as pio

In [2]:
# import config from paper_figures/ (parent of this notebook's folder)
import sys
from pathlib import Path

try:
    _paper_figures = Path(__file__).resolve().parent.parent
except NameError:
    # Jupyter: __file__ is undefined; cwd is usually paper_figures or pumping/
    _cwd = Path.cwd()
    _paper_figures = _cwd.parent if _cwd.name == "pumping" else _cwd

sys.path.insert(0, str(_paper_figures))
import utils

In [3]:
ensemble_name = "baseline"
ensemble_member = "baseline"
no_flow_barrier = utils.read_simulation_data(ensemble_name, ensemble_member, utils.DOMAIN)

/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'cfradial1' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'furuno' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'gamic' loading failed:
No module named 'xarray.core.merge'
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
ERROR 1: PROJ: proj_create_from_database: Open of /glade/work/bwest/conda-envs/droughts/share/proj failed
/glade/work/bwest/conda-envs/droughts/lib/python3.11/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: 

In [4]:
# ensemble_member = "pumping_0_00001"
# data = utils.read_simulation_data(ensemble_name, ensemble_member, utils.DOMAIN)
# data.info()

In [5]:
# total_storage = data.storage.sel(time=slice(None, None, 8760)).sum(dim=["x", "y","z"])
total_storage_baseline = no_flow_barrier.storage.sel(time=slice(None, None, 8760)).sum(dim=["x", "y","z"])

In [6]:
px.line(total_storage_baseline)

In [7]:
import plotly.graph_objects as go

# Compute yearly PME total from the same domain used by this baseline run
raw_nc_path = Path(json.loads(Path(
    f"/glade/derecho/scratch/bwest/drought-ensemble/domains/{utils.DOMAIN}/processed_full_runs/{ensemble_name}/{ensemble_member}/file_locations.json"
).read_text())[0])
run_dir = raw_nc_path.parent
pme = pf.read_pfb(str(run_dir / "pme.pfb"))

if pme.ndim != 3:
    raise ValueError(f"Unexpected pme.pfb shape: {pme.shape}")

# Domain mask (active cells only)
mask_2d = no_flow_barrier.mask.isel(time=0, z=0).values > 0

# pme.pfb can be stored with a single active z layer; collapse if needed
layer_activity = np.sum(np.abs(pme), axis=(1, 2))
active_layers = np.where(layer_activity > 0)[0]
if active_layers.size == 1:
    pme_2d = pme[active_layers[0], :, :]
else:
    pme_2d = np.sum(pme, axis=0)

# Use the same grid spacing as the yearly PME notebook.
# Processed NetCDF x/y are index coordinates (0,1,2,...) not physical meters.
DX = 1000.0
DY = 1000.0
HOURS_PER_YEAR = 365.0 * 24.0

yearly_pme_total_m3 = float(
    np.sum(np.where(mask_2d, pme_2d, 0.0)) * DX * DY * HOURS_PER_YEAR
)
if yearly_pme_total_m3 == 0:
    raise ValueError("Domain-integrated yearly PME is zero; cannot compute fraction.")

# Year-to-year storage change (finite-difference derivative over each 1-year interval)
years = total_storage_baseline.time.values
delta_storage_m3 = np.diff(total_storage_baseline.values)
interval_mid_years = 0.5 * (years[:-1] + years[1:])
fraction_of_yearly_pme = delta_storage_m3 / yearly_pme_total_m3

graph = go.Figure()
graph.add_trace(go.Scatter(
    x=interval_mid_years,
    y=fraction_of_yearly_pme,
    mode='lines+markers',
    name='Baseline annual ΔS / yearly PME'
))
graph.add_hline(y=0.0, line_width=1, line_color='black')
graph.update_layout(
    xaxis_title="year",
    yaxis_title="Storage change / yearly PME (-)",
    title="Annual storage change as fraction of yearly PME"
)

graph.show()

# Consistency check: this should be 1.0 for uniform 1-year spacing.
dt_years = np.diff(years)
backward_derivative_m3_per_year = delta_storage_m3 / dt_years
print(f"Using yearly PME total = {yearly_pme_total_m3:,.3e} m^3/year")
print(
    f"Fraction range: {np.nanmin(fraction_of_yearly_pme):.4f} to "
    f"{np.nanmax(fraction_of_yearly_pme):.4f}"
)
print(
    "corr(ΔS, dS/dt over interval) = "
    f"{np.corrcoef(delta_storage_m3, backward_derivative_m3_per_year)[0, 1]:.6f}"
)

# Extra sanity checks for "by-eye" slope comparisons on the first graph.
start_end_single_year_ratio = abs(delta_storage_m3[-1]) / abs(delta_storage_m3[0])
start_end_five_year_ratio = (
    abs(np.mean(delta_storage_m3[-5:])) / abs(np.mean(delta_storage_m3[:5]))
)
print(
    "|ΔS_last_year| / |ΔS_first_year| = "
    f"{start_end_single_year_ratio:.3f}"
)
print(
    "|mean(ΔS last 5y)| / |mean(ΔS first 5y)| = "
    f"{start_end_five_year_ratio:.3f}"
)


Using yearly PME total = 1.523e+09 m^3/year
Fraction range: -0.0075 to -0.0033
corr(ΔS, dS/dt over interval) = 1.000000
|ΔS_last_year| / |ΔS_first_year| = 0.436
|mean(ΔS last 5y)| / |mean(ΔS first 5y)| = 0.529


In [8]:
ensemble_name = "pumping_ensemble"
flow_barrier = utils.read_simulation_data(ensemble_name, ensemble_member, utils.DOMAIN_2)
flow_barrier.info()

/glade/derecho/scratch/bwest/drought-ensemble/analysis/paper_figures/utils.py:35: FutureWarning:

In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.



xarray.Dataset {
dimensions:
	time = 157680 ;
	z = 10 ;
	y = 136 ;
	x = 70 ;

variables:
	float64 pressure(time, z, y, x) ;
	float64 saturation(time, z, y, x) ;
	float64 evaptrans(time, z, y, x) ;
	float64 overland_bc_flux(time, y, x) ;
	float64 mask(time, z, y, x) ;
	float64 mannings(time, y, x) ;
	float64 porosity(time, z, y, x) ;
	float64 specific_storage(time, z, y, x) ;
	float64 DZ_Multiplier(time, z, y, x) ;
	float64 slopex(time, y, x) ;
	float64 slopey(time, y, x) ;
	float64 perm_x(time, z, y, x) ;
	float64 perm_y(time, z, y, x) ;
	float64 perm_z(time, z, y, x) ;
	float64 overland_flow(time, y, x) ;
	float64 storage(time, z, y, x) ;
	float64 time(time) ;

// global attributes:
}

In [9]:
#plot the baseline outlet flow
from typing import Any


px.line(flow_barrier.overland_flow.isel( x=utils.OUTLET_X, y=utils.OUTLET_Y, time=slice(8760*10, 8760*11,24)))
# px.line(baseline.outlet_flow)

In [10]:
px.line(no_flow_barrier.overland_flow.isel( x=utils.OUTLET_X, y=utils.OUTLET_Y, time=slice(8760*10, 8760*11,24)))


In [11]:
px.imshow(no_flow_barrier.overland_flow.isel(time=8760*10))

In [12]:
px.imshow(flow_barrier.overland_flow.isel(time=8760*10))

In [13]:
def get_wtd(data, time_index=0):
    pressure_snapshot = data.pressure.isel(time=time_index).values
    saturation_snapshot = data.saturation.isel(time=time_index).values
    dz_xr = data.DZ_Multiplier.isel(time=time_index)

    if dz_xr.ndim == 1:
        dz_1d = np.asarray(dz_xr.values, dtype=float)
    else:
        # (z, y, x) in processed runs: use one column (typical when dz does not vary horizontally)
        dz_1d = np.asarray(dz_xr.isel(x=20, y=20).values, dtype=float)
        dz_1d = dz_1d*200.

    print("pressure:", pressure_snapshot.shape, "saturation:", saturation_snapshot.shape, "dz:", dz_1d.shape)

    wtd = calculate_water_table_depth(pressure_snapshot, saturation_snapshot, dz_1d)
    print("water table depth (ny, nx):", wtd.shape)
    # apply the mask to the wtd
    mask = data.mask.isel(time=0, z=0).values
    #normalize the mask to 0-1
    mask = mask/mask.max()
    wtd = wtd*mask

    return wtd

In [14]:
wtd_flow_barrier = get_wtd(flow_barrier, 8760*10)
wtd_no_flow_barrier = get_wtd(no_flow_barrier, 8760*10)
px.imshow(wtd_no_flow_barrier-wtd_flow_barrier)



pressure: (10, 136, 70) saturation: (10, 136, 70) dz: (10,)
water table depth (ny, nx): (136, 70)
pressure: (10, 136, 70) saturation: (10, 136, 70) dz: (10,)
water table depth (ny, nx): (136, 70)
